In [1]:
!pip install huggingface_hub

# Cleaning extra files

In [7]:
"""
Fetches all mp3 filenames from Lipi-Ghor-bn-882-SSTT/data on HuggingFace
and writes a CSV with video_id and youtube_url columns.

Requirements:
    pip install huggingface_hub
"""
_
from huggingface_hub import list_repo_tree
import csv
import os

REPO_ID = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"
FOLDER = "diarization_results"
OUTPUT_CSV = "lipi_ghor_diarization_results_ids.csv"

print(f"Fetching file list from {REPO_ID}/{FOLDER} ...")

files = list_repo_tree(
    repo_id=REPO_ID,
    path_in_repo=FOLDER,
    repo_type="dataset",
    recursive=False,
)

rows = []
for f in files:
    name = os.path.basename(f.path)
    if not name.endswith(".json"):
        continue
    video_id = name[:-12]  # strip .mp3
    youtube_url = f"https://www.youtube.com/watch?v={video_id}"
    rows.append({"video_id": video_id, "youtube_url": youtube_url})

rows.sort(key=lambda x: x["video_id"])

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["video_id", "youtube_url"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Done. {len(rows)} videos written to {OUTPUT_CSV}")

Fetching file list from Sanjidh090/Lipi-Ghor-bn-882-SSTT/diarization_results ...
Done. 1019 videos written to lipi_ghor_diarization_results_ids.csv


In [4]:
"""
Finds video IDs present in data/diarization_results but missing
from diarization_transcription_final.

Requirements:
    pip install huggingface_hub
"""

from huggingface_hub import list_repo_tree
import csv
import os

REPO_ID = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"
REPO_TYPE = "dataset"

FOLDERS = {
    "data":                          ".mp3",
    "diarization_results":           ".json",
    "diarization_transcription_final": ".json",
}

def get_ids(folder, ext):
    files = list_repo_tree(repo_id=REPO_ID, path_in_repo=folder,
                           repo_type=REPO_TYPE, recursive=False)
    ids = set()
    for f in files:
        name = os.path.basename(f.path)
        if name.endswith(ext):
            # handles names like <video_id>.json or <video_id>_diarization.json etc.
            vid = name.replace(ext, "").split("_")[0]
            ids.add(vid)
    print(f"  {folder}: {len(ids)} IDs")
    return ids

print("Fetching IDs from all three folders...")
data_ids      = get_ids("data",                            ".mp3")
diar_ids      = get_ids("diarization_results",             ".json")
trans_ids     = get_ids("diarization_transcription_final", ".json")

# Sanity checks
only_in_data_not_diar  = data_ids  - diar_ids
only_in_diar_not_data  = diar_ids  - data_ids
missing_transcription  = diar_ids  - trans_ids   # has diar, no final transcript
extra_in_trans         = trans_ids - diar_ids    # in transcript but not diar (shouldn't happen)

print(f"\n--- Summary ---")
print(f"data:                          {len(data_ids)}")
print(f"diarization_results:           {len(diar_ids)}")
print(f"diarization_transcription_final: {len(trans_ids)}")
print(f"\nIn data but NOT diarization:   {len(only_in_data_not_diar)}")
print(f"In diarization but NOT data:   {len(only_in_diar_not_data)}")
print(f"Have diarization but NO final transcription: {len(missing_transcription)}")
print(f"In transcription but NOT diarization:        {len(extra_in_trans)}")

# Write missing transcription IDs to CSV
output = "missing_transcriptions.csv"
missing_sorted = sorted(missing_transcription)
with open(output, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["video_id", "youtube_url"])
    writer.writeheader()
    for vid in missing_sorted:
        writer.writerow({"video_id": vid,
                         "youtube_url": f"https://www.youtube.com/watch?v={vid}"})

print(f"\nMissing transcription IDs written to: {output}")
if missing_sorted:
    print("Missing IDs:")
    for v in missing_sorted:
        print(f"  {v}")

Fetching IDs from all three folders...
  data: 996 IDs
  diarization_results: 996 IDs
  diarization_transcription_final: 996 IDs

--- Summary ---
data:                          996
diarization_results:           996
diarization_transcription_final: 996

In data but NOT diarization:   0
In diarization but NOT data:   0
Have diarization but NO final transcription: 0
In transcription but NOT diarization:        0

Missing transcription IDs written to: missing_transcriptions.csv


In [ ]:
"""
Deletes mp3 files from data/ folder on HuggingFace for video IDs
that have diarization but no final transcription.

Requirements:
    pip install huggingface_hub
"""

from huggingface_hub import list_repo_tree, delete_file
import os

REPO_ID   = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"
REPO_TYPE = "dataset"
HF_TOKEN  = "HF_TOKEN"   # <-- paste your token

FOLDERS = {
    "data":                            ".mp3",
    "diarization_results":             ".json",
    "diarization_transcription_final": ".json",
}

def get_ids_and_paths(folder, ext):
    files = list_repo_tree(repo_id=REPO_ID, path_in_repo=folder,
                           repo_type=REPO_TYPE, recursive=False,
                           token=HF_TOKEN)
    id_to_path = {}
    for f in files:
        name = os.path.basename(f.path)
        if name.endswith(ext):
            vid = name.replace(ext, "").split("_")[0]
            id_to_path[vid] = f.path
    print(f"  {folder}: {len(id_to_path)} files")
    return id_to_path

print("Fetching file lists...")
data_map  = get_ids_and_paths("data",                            ".mp3")
diar_map  = get_ids_and_paths("diarization_results",             ".json")
trans_map = get_ids_and_paths("diarization_transcription_final", ".json")

missing = sorted(set(diar_map.keys()) - set(trans_map.keys()))
print(f"\n{len(missing)} videos to delete from data/:")
for v in missing:
    print(f"  {v}")

if not missing:
    print("Nothing to delete.")
    exit()

confirm = input(f"\nDelete these {len(missing)} files from data/? [y/N]: ").strip().lower()
if confirm != "y":
    print("Aborted.")
    exit()

deleted, skipped = [], []
for vid in missing:
    if vid not in data_map:
        print(f"  SKIP (not in data/): {vid}")
        skipped.append(vid)
        continue
    path = data_map[vid]
    try:
        delete_file(
            path_in_repo=path,
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            token=HF_TOKEN,
            commit_message=f"Remove unmatched video: {vid}",
        )
        print(f"  DELETED: {path}")
        deleted.append(vid)
    except Exception as e:
        print(f"  ERROR ({vid}): {e}")
        skipped.append(vid)

print(f"\nDone. Deleted: {len(deleted)}, Skipped/errors: {len(skipped)}")

Fetching file lists...
  data: 996 files
  diarization_results: 996 files
  diarization_transcription_final: 996 files

0 videos to delete from data/:
Nothing to delete.



Delete these 0 files from data/? [y/N]:  y



Done. Deleted: 0, Skipped/errors: 0


In [6]:
"""
Fetches all mp3 filenames from Lipi-Ghor-bn-882-SSTT/data on HuggingFace
and writes a CSV with video_id and youtube_url columns.

Requirements:
    pip install huggingface_hub
"""

from huggingface_hub import list_repo_tree
import csv
import os

REPO_ID = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"
FOLDER = "data"
OUTPUT_CSV = "lipi_ghor_video_ids.csv"

print(f"Fetching file list from {REPO_ID}/{FOLDER} ...")

files = list_repo_tree(
    repo_id=REPO_ID,
    path_in_repo=FOLDER,
    repo_type="dataset",
    recursive=False,
)

rows = []
for f in files:
    name = os.path.basename(f.path)
    if not name.endswith(".mp3"):
        continue
    video_id = name[:-4]  # strip .mp3
    youtube_url = f"https://www.youtube.com/watch?v={video_id}"
    rows.append({"video_id": video_id, "youtube_url": youtube_url})

rows.sort(key=lambda x: x["video_id"])

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["video_id", "youtube_url"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Done. {len(rows)} videos written to {OUTPUT_CSV}")

Fetching file list from Sanjidh090/Lipi-Ghor-bn-882-SSTT/data ...
Done. 1019 videos written to lipi_ghor_video_ids.csv


In [ ]:
"""
Deletes files from diarization_results/ for video IDs NOT present in data/.
Uses data/ as ground truth (1019 files = the correct set).

Requirements:
    pip install huggingface_hub
"""

from huggingface_hub import list_repo_tree, CommitOperationDelete, create_commit
import os

REPO_ID   = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"
REPO_TYPE = "dataset"
HF_TOKEN  = "HF_TOKEN"   # <-- paste your token

def get_id_map(folder, ext, strip_suffix):
    files = list(list_repo_tree(repo_id=REPO_ID, path_in_repo=folder,
                                repo_type=REPO_TYPE, recursive=False,
                                token=HF_TOKEN))
    id_to_path = {}
    for f in files:
        name = os.path.basename(f.path)
        if name.endswith(ext):
            vid = name[:-len(strip_suffix)]
            id_to_path[vid] = f.path
    print(f"  {folder}: {len(id_to_path)} files")
    return id_to_path

print("Fetching file lists...")
data_map = get_id_map("data",                ".mp3",  ".mp3")
diar_map = get_id_map("diarization_results", ".json", "_output.json")

# Ground truth = data/. Delete anything in diar not in data.
extras = sorted(set(diar_map.keys()) - set(data_map.keys()))
print(f"\n{len(extras)} extra IDs in diarization_results/ not in data/:")
for v in extras:
    print(f"  {v}  ->  {diar_map[v]}")

if not extras:
    print("Nothing to delete.")
    exit()

to_delete = [diar_map[vid] for vid in extras]

confirm = input(f"\nDelete these {len(to_delete)} json files in one commit? [y/N]: ").strip().lower()
if confirm != "y":
    print("Aborted.")
    exit()

operations = [CommitOperationDelete(path_in_repo=p) for p in to_delete]

create_commit(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=HF_TOKEN,
    operations=operations,
    commit_message=f"Remove {len(extras)} unmatched entries from diarization_results/",
)

print(f"\nDone. {len(to_delete)} files deleted.")

Fetching file lists...
  data: 1019 files
  diarization_results: 1019 files

0 extra IDs in diarization_results/ not in data/:
Nothing to delete.



Delete these 0 json files in one commit? [y/N]:  y


No files have been modified since last commit. Skipping to prevent empty commit.



Done. 0 files deleted.


# A new part is here,,,we will bring our srcs

In [2]:
!wget https://raw.githubusercontent.com/Sanjidx090/lp/refs/heads/main/src/video_ids_metadata_Bengali%20-%20Sanjid.csv

--2026-03-19 16:47:02--  https://raw.githubusercontent.com/Sanjidx090/lp/refs/heads/main/src/video_ids_metadata_Bengali%20-%20Sanjid.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63697 (62K) [text/plain]
Saving to: ‘video_ids_metadata_Bengali - Sanjid.csv.1’

video_ids_metadata_ 100%[===================>]  62.20K  --.-KB/s    in 0.01s   

2026-03-19 16:47:02 (5.23 MB/s) - ‘video_ids_metadata_Bengali - Sanjid.csv.1’ saved [63697/63697]



In [3]:
!wget https://raw.githubusercontent.com/Sanjidx090/lp/refs/heads/main/src/video_ids_metadata_Bengali%20-%20Talkshow.csv

--2026-03-19 16:47:02--  https://raw.githubusercontent.com/Sanjidx090/lp/refs/heads/main/src/video_ids_metadata_Bengali%20-%20Talkshow.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 132229 (129K) [text/plain]
Saving to: ‘video_ids_metadata_Bengali - Talkshow.csv’

video_ids_metadata_ 100%[===================>] 129.13K  --.-KB/s    in 0.02s   

2026-03-19 16:47:03 (5.32 MB/s) - ‘video_ids_metadata_Bengali - Talkshow.csv’ saved [132229/132229]



In [4]:
!wget https://raw.githubusercontent.com/Sanjidx090/lp/refs/heads/main/src/video_ids_metadata_Bengali%20-%20Team_Villagers.csv

--2026-03-19 16:47:05--  https://raw.githubusercontent.com/Sanjidx090/lp/refs/heads/main/src/video_ids_metadata_Bengali%20-%20Team_Villagers.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 114274 (112K) [text/plain]
Saving to: ‘video_ids_metadata_Bengali - Team_Villagers.csv’

video_ids_metadata_ 100%[===================>] 111.60K  --.-KB/s    in 0.02s   

2026-03-19 16:47:05 (7.15 MB/s) - ‘video_ids_metadata_Bengali - Team_Villagers.csv’ saved [114274/114274]



In [5]:
!wget https://raw.githubusercontent.com/Sanjidx090/lp/refs/heads/main/src/video_ids_metadata_Bengali%20-%20audio.csv

--2026-03-19 16:47:08--  https://raw.githubusercontent.com/Sanjidx090/lp/refs/heads/main/src/video_ids_metadata_Bengali%20-%20audio.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 70032 (68K) [text/plain]
Saving to: ‘video_ids_metadata_Bengali - audio.csv’

video_ids_metadata_ 100%[===================>]  68.39K  --.-KB/s    in 0.01s   

2026-03-19 16:47:08 (5.44 MB/s) - ‘video_ids_metadata_Bengali - audio.csv’ saved [70032/70032]



In [11]:
"""
Takes lipi_ghor_video_ids.csv (1019 IDs) as master list.
Looks up each video_id across the 4 source CSVs and collects metadata.
Outputs both CSV and XLSX.

Requirements:
    pip install pandas openpyxl
"""

import pandas as pd
import csv
import os

MASTER   = "lipi_ghor_video_ids.csv"
OUTPUT_CSV  = "lipi_ghor_metadata.csv"
OUTPUT_XLSX = "lipi_ghor_metadata.xlsx"

SOURCE_FILES = [
    "video_ids_metadata_Bengali - Sanjid.csv",
    "video_ids_metadata_Bengali - Talkshow.csv",
    "video_ids_metadata_Bengali - Team_Villagers.csv",
    "video_ids_metadata_Bengali - audio.csv",
]

COL_MAP = {
    "duration(sec)": "duration_sec",
    "domain":        "category",
    "link":          "video_url",
}

# ── Load master IDs ──────────────────────────────────────────────
master = pd.read_csv(MASTER)
master_ids = set(master["video_id"].astype(str).str.strip())
print(f"Master IDs: {len(master_ids)}")

# ── Build lookup dict: video_id → metadata row ──────────────────
lookup = {}  # video_id -> dict

for path in SOURCE_FILES:
    if not os.path.exists(path):
        print(f"  MISSING: {path} — skipping")
        continue

    # Use QUOTE_NONE to handle malformed quoting in Team_Villagers
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f, quoting=csv.QUOTE_NONE, escapechar="\\")
        headers = None
        for i, row in enumerate(reader):
            if i == 0:
                headers = [h.strip() for h in row]
                continue
            if len(row) != len(headers):
                continue  # skip malformed lines
            rows.append(dict(zip(headers, [c.strip() for c in row])))

    df = pd.DataFrame(rows)
    df.rename(columns=COL_MAP, inplace=True)

    # Normalise video_id col
    if "video_id" not in df.columns:
        print(f"  NO video_id col in {path}, cols: {list(df.columns)}")
        continue

    df["video_id"] = df["video_id"].astype(str).str.strip()
    matched = df[df["video_id"].isin(master_ids)]
    print(f"  {os.path.basename(path)}: {len(matched)} matches out of {len(df)} rows")

    for _, row in matched.iterrows():
        vid = row["video_id"]
        if vid not in lookup:          # first match wins
            lookup[vid] = row.to_dict()

print(f"\nTotal IDs with metadata found: {len(lookup)}")
print(f"IDs with NO metadata:          {len(master_ids) - len(lookup)}")

# ── Build output dataframe ───────────────────────────────────────
FINAL_COLS = ["video_id", "channel_id", "channel_name",
              "duration_sec", "category", "video_url"]

records = []
for vid in sorted(master_ids):
    if vid in lookup:
        row = lookup[vid]
        records.append({
            "video_id":    vid,
            "channel_id":  row.get("channel_id", ""),
            "channel_name":row.get("channel_name", ""),
            "duration_sec":row.get("duration_sec", ""),
            "category":    row.get("category", ""),
            "video_url":   row.get("video_url", f"https://www.youtube.com/watch?v={vid}"),
        })
    else:
        records.append({
            "video_id":    vid,
            "channel_id":  "",
            "channel_name":"",
            "duration_sec":"",
            "category":    "",
            "video_url":   f"https://www.youtube.com/watch?v={vid}",
        })

result = pd.DataFrame(records, columns=FINAL_COLS)

result.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
result.to_excel(OUTPUT_XLSX, index=False)

print(f"\nSaved: {OUTPUT_CSV}")
print(f"Saved: {OUTPUT_XLSX}")
print(f"\nCategory breakdown:\n{result['category'].value_counts().to_string()}")

Master IDs: 1019
  video_ids_metadata_Bengali - Sanjid.csv: 264 matches out of 538 rows
  video_ids_metadata_Bengali - Talkshow.csv: 131 matches out of 1180 rows
  video_ids_metadata_Bengali - Team_Villagers.csv: 431 matches out of 959 rows
  video_ids_metadata_Bengali - audio.csv: 231 matches out of 582 rows

Total IDs with metadata found: 1001
IDs with NO metadata:          18

Saved: lipi_ghor_metadata.csv
Saved: lipi_ghor_metadata.xlsx

Category breakdown:
category
Audio-book                              230
Talk-show                               121
podcast                                  33
movie                                    30
cartoon                                  23
food vlog                                21
News Presentation                        21
Waz                                      20
drama                                    20
natok                                    20
zatrapala                                18
                                         1

In [9]:
missing = sorted(master_ids - set(lookup.keys()))
for v in missing:
    print(v)

B5hbaTGbyUk
I1zFxRTNdhU
IKeUBaHFX9k
NczfOOsYl48
QP9Yw_jjOGQ
U210Vg7ikiA
Zs1-cNPKfAM
bJPfIlUmpEE
dTc8PKo0MZY
e6T3a4tEvB8
eWsaNxD4R7s
g7mH0bqlFms
jzS_2zItbAI
kD64SxX3UhA
lM7CXSyknEI
mG7ByReadRE
svhpW9_mWIM
z5khvHT6reI


In [13]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GOOGLE_API_KEY")


In [10]:
CATEGORY_MAP = {
    "waz": "Waz",
    "waz / islamic sermon": "Waz",
    "waz / islamic talk": "Waz",
    "talk show": "Talk-show",
    "talkshow": "Talk-show",
    "political talk show": "Talk-show",
    "natok": "Natok",
    "bangla natok": "Natok",
    "short natok": "Natok",
    "comedy natok": "Natok",
    "audiobook": "Audio-book",
    "audiobook / storytelling": "Audio-book",
    "audiobook / story": "Audio-book",
    # add more as needed
}

result["category"] = result["category"].str.strip().str.lower().map(
    lambda x: CATEGORY_MAP.get(x, x)
)

In [12]:
import subprocess, json

def get_yt_metadata(video_id):
    url = f"https://www.youtube.com/watch?v={video_id}"
    result = subprocess.run(
        ["yt-dlp", "--dump-json", "--no-download", url],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        return None
    data = json.loads(result.stdout)
    return {
        "video_id":    video_id,
        "channel_id":  data.get("channel_id", ""),
        "channel_name":data.get("channel", ""),
        "duration_sec":data.get("duration", ""),
        "category":    data.get("categories", [""])[0],
        "video_url":   f"https://www.youtube.com/watch?v={video_id}",
    }

In [15]:
!pip install isodate

In [17]:
"""
Fills metadata for the 18 missing video IDs using YouTube Data API v3,
then patches them into the existing lipi_ghor_metadata.csv and .xlsx.

Requirements:
    pip install pandas openpyxl requests isodate
"""

import pandas as pd
import requests
import isodate
import os

API_KEY     = secret_value_0   # <-- paste your YT API key
METADATA_CSV  = "lipi_ghor_metadata.csv"
OUTPUT_CSV    = "lipi_ghor_metadata.csv"      # overwrite in place
OUTPUT_XLSX   = "lipi_ghor_metadata.xlsx"

MISSING_IDS = [
    "B5hbaTGbyUk", "I1zFxRTNdhU", "IKeUBaHFX9k", "NczfOOsYl48",
    "QP9Yw_jjOGQ", "U210Vg7ikiA", "Zs1-cNPKfAM", "bJPfIlUmpEE",
    "dTc8PKo0MZY", "e6T3a4tEvB8", "eWsaNxD4R7s", "g7mH0bqlFms",
    "jzS_2zItbAI", "kD64SxX3UhA", "lM7CXSyknEI", "mG7ByReadRE",
    "svhpW9_mWIM", "z5khvHT6reI",
]

def fetch_batch(video_ids, api_key):
    """Fetch metadata for up to 50 IDs in one API call."""
    r = requests.get(
        "https://www.googleapis.com/youtube/v3/videos",
        params={
            "id":   ",".join(video_ids),
            "part": "snippet,contentDetails",
            "key":  api_key,
        }
    )
    r.raise_for_status()
    return r.json().get("items", [])

def parse_duration(iso_duration):
    """Convert ISO 8601 duration (PT1H2M3S) to total seconds."""
    try:
        return int(isodate.parse_duration(iso_duration).total_seconds())
    except:
        return ""

# ── Fetch from YouTube API ───────────────────────────────────────
print(f"Fetching metadata for {len(MISSING_IDS)} video IDs...")
items = fetch_batch(MISSING_IDS, API_KEY)
print(f"  API returned {len(items)} results")

fetched = {}
for item in items:
    vid     = item["id"]
    snippet = item["snippet"]
    dur_iso = item["contentDetails"].get("duration", "")
    fetched[vid] = {
        "video_id":    vid,
        "channel_id":  snippet.get("channelId", ""),
        "channel_name":snippet.get("channelTitle", ""),
        "duration_sec":parse_duration(dur_iso),
        "category":    "",   # fill manually or leave blank
        "video_url":   f"https://www.youtube.com/watch?v={vid}",
    }

not_found = set(MISSING_IDS) - set(fetched.keys())
if not_found:
    print(f"  Not found (deleted/private): {not_found}")

# ── Patch into existing CSV ──────────────────────────────────────
df = pd.read_csv(METADATA_CSV)

for vid, meta in fetched.items():
    mask = df["video_id"] == vid
    if mask.any():
        for col, val in meta.items():
            if col != "video_id":
                df.loc[mask, col] = val
        print(f"  PATCHED: {vid} | {meta['channel_name']} | {meta['duration_sec']}s")
    else:
        print(f"  WARNING: {vid} not in master CSV (shouldn't happen)")

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
df.to_excel(OUTPUT_XLSX, index=False)

print(f"\nDone. Saved {OUTPUT_CSV} and {OUTPUT_XLSX}")
print(f"\nRemaining empty rows:")
empty = df[df["channel_id"] == ""]["video_id"].tolist()
print(empty if empty else "  None — all filled!")

Fetching metadata for 18 video IDs...
  API returned 18 results
  PATCHED: B5hbaTGbyUk | AudioKothon with RAJIA | 3088s
  PATCHED: I1zFxRTNdhU | AudioKothon with RAJIA | 4509s
  PATCHED: IKeUBaHFX9k | AudioKothon with RAJIA | 4088s
  PATCHED: NczfOOsYl48 | AudioKothon with RAJIA | 3202s
  PATCHED: QP9Yw_jjOGQ | AudioKothon with RAJIA | 2604s
  PATCHED: U210Vg7ikiA | AudioKothon with RAJIA | 3210s
  PATCHED: Zs1-cNPKfAM | AudioKothon with RAJIA | 2370s
  PATCHED: bJPfIlUmpEE | AudioKothon with RAJIA | 3582s
  PATCHED: dTc8PKo0MZY | AudioKothon with RAJIA | 3170s
  PATCHED: e6T3a4tEvB8 | AudioKothon with RAJIA | 3153s
  PATCHED: eWsaNxD4R7s | AudioKothon with RAJIA | 3431s
  PATCHED: g7mH0bqlFms | AudioKothon with RAJIA | 3777s
  PATCHED: jzS_2zItbAI | AudioKothon with RAJIA | 2906s
  PATCHED: kD64SxX3UhA | AudioKothon with RAJIA | 2738s
  PATCHED: lM7CXSyknEI | AudioKothon with RAJIA | 3001s
  PATCHED: mG7ByReadRE | AudioKothon with RAJIA | 4671s
  PATCHED: svhpW9_mWIM | AudioKothon wit

In [18]:
df = pd.read_csv("lipi_ghor_metadata.csv")

df.loc[df["channel_name"] == "AudioKothon with RAJIA", "category"] = "Audio-book"

df.to_csv("lipi_ghor_metadata.csv", index=False)
df.to_excel("lipi_ghor_metadata.xlsx", index=False)
print("Done.")

Done.


In [19]:
"""
Category distribution analysis for lipi_ghor_metadata.csv
"""

import pandas as pd

df = pd.read_csv("lipi_ghor_metadata.csv")
total = len(df)

# ── Normalize categories ─────────────────────────────────────────
NORM = {
    # Waz
    "waz": "Waz",
    "waz / islamic sermon": "Waz",
    "waz / islamic talk": "Waz",
    "islamic lectures": "Waz",
    "islamic talk": "Waz",
    "islamic lecture (bangla dub/run)": "Waz",
    "islamic stories": "Waz",
    "speech islamik": "Waz",
    # Talk-show
    "talk show": "Talk-show",
    "talkshow": "Talk-show",
    "talk-show": "Talk-show",
    "political talk show": "Talk-show",
    "reality talk show": "Talk-show",
    "entertainment talk show": "Talk-show",
    "talk & social commentary": "Talk-show",
    "live talk": "Talk-show",
    "sylheti talk": "Talk-show",
    # Podcast
    "personal podcast": "Podcast",
    "conversational podcast": "Podcast",
    "story podcast": "Podcast",
    "entertainment podcast": "Podcast",
    "technology podcast": "Podcast",
    "business podcast": "Podcast",
    "mental health podcast": "Podcast",
    "music podcast": "Podcast",
    "story & talk podcast": "Podcast",
    # Audio-book
    "audiobook": "Audio-book",
    "audiobook / storytelling": "Audio-book",
    "audiobook / story": "Audio-book",
    "audio-book": "Audio-book",
    "bangla puthi": "Audio-book",
    # Natok / Drama
    "natok": "Natok",
    "bangla natok": "Natok",
    "short natok": "Natok",
    "comedy natok": "Natok",
    "web natok": "Natok",
    "natok / acting content": "Natok",
    "natok / telefilm": "Natok",
    "natok / short film": "Natok",
    "natok archive": "Natok",
    "drama": "Natok",
    "audio drama": "Natok",
    "tv drama": "Natok",
    "tv & stage drama": "Natok",
    "movies / drama": "Natok",
    "sylheti drama": "Natok",
    "chittagonian drama": "Natok",
    "stage drama": "Natok",
    # Movie
    "movie": "Movie",
    "bangla cinema": "Movie",
    "bangla cinema explanation": "Movie",
    "kolkata bangla movie": "Movie",
    "web series": "Movie",
    "movies / web series": "Movie",
    "web film": "Movie",
    # News
    "news": "News",
    "news presentation": "News",
    "international news": "News",
    "dbc news": "News",
    "investigative & political commentary": "News",
    # Education
    "education": "Education",
    "online class": "Education",
    "online learning": "Education",
    "web development": "Education",
    "programming & tech": "Education",
    "programming education": "Education",
    "programming tutorials": "Education",
    "gk jobs": "Education",
    "bcs preli": "Education",
    "e-learning platform": "Education",
    # Political
    "political commentary / analysis": "Political",
    "political analysis": "Political",
    "political analysis / opinion": "Political",
    "political talk": "Political",
    "political & social talk": "Political",
    "debate": "Political",
    "regional language debate": "Political",
    "parliament session": "Political",
    # Music
    "music": "Music",
    "music & talk": "Music",
    "drama & music": "Music",
    "sylheti music": "Music",
    "kirton": "Music",
    "palagan": "Music",
    "poem": "Music",
    "poetry": "Music",
    # Vlog / Casual
    "food vlog": "Vlog",
    "tour vlog": "Vlog",
    "vlogs & informal talk": "Vlog",
    "informal discussion": "Vlog",
    "food & casual talk": "Vlog",
    # Comedy / Satire
    "satire / social commentary": "Comedy / Satire",
    "satire / talk": "Comedy / Satire",
    "comedy & acting": "Comedy / Satire",
    "rangpuri comedy": "Comedy / Satire",
    "zatrapala": "Comedy / Satire",
    "ha show": "Comedy / Satire",
    "what a show": "Comedy / Satire",
    # Cartoon
    "cartoon": "Cartoon",
}

df["category_norm"] = (
    df["category"]
    .fillna("")
    .str.strip()
    .str.lower()
    .map(lambda x: NORM.get(x, x.title() if x else "Unknown"))
)

# ── Print distribution ───────────────────────────────────────────
counts = df["category_norm"].value_counts()

print("=" * 50)
print(f"  LIPI-GHOR CATEGORY DISTRIBUTION  (n={total})")
print("=" * 50)
print(f"{'Category':<35} {'Count':>6}  {'%':>6}")
print("-" * 50)
for cat, cnt in counts.items():
    pct = cnt / total * 100
    bar = "█" * int(pct / 2)
    print(f"{cat:<35} {cnt:>6}  {pct:>5.1f}%  {bar}")
print("-" * 50)
print(f"{'TOTAL':<35} {total:>6}  100.0%")
print(f"\nDistinct categories: {len(counts)}")
print(f"Unknown / unmapped:  {counts.get('Unknown', 0)}")

# Show unmapped originals if any
unmapped = df[df["category_norm"] == "Unknown"]["category"].value_counts()
if not unmapped.empty:
    print("\nUnmapped original values:")
    for v, c in unmapped.items():
        print(f"  '{v}' ({c}x)")

  LIPI-GHOR CATEGORY DISTRIBUTION  (n=1019)
Category                             Count       %
--------------------------------------------------
Audio-book                             273   26.8%  █████████████
Talk-show                              157   15.4%  ███████
Natok                                   95    9.3%  ████
Movie                                   80    7.9%  ███
Waz                                     65    6.4%  ███
Podcast                                 60    5.9%  ██
Comedy / Satire                         43    4.2%  ██
Vlog                                    42    4.1%  ██
Education                               39    3.8%  █
Music                                   36    3.5%  █
News                                    35    3.4%  █
Political                               29    2.8%  █
Cartoon                                 23    2.3%  █
Tv Show                                 10    1.0%  
Bangla Comentary                         7    0.7%  
Literature Discuss

In [20]:
"""
Applies full category normalization to lipi_ghor_metadata.csv and .xlsx.
Collapses all spelling/casing variants into clean canonical categories.
"""

import pandas as pd

INPUT_CSV   = "lipi_ghor_metadata.csv"
OUTPUT_CSV  = "lipi_ghor_metadata.csv"
OUTPUT_XLSX = "lipi_ghor_metadata.xlsx"

NORM = {
    # ── Waz ──────────────────────────────────────────────────────
    "waz": "Waz",
    "waz / islamic sermon": "Waz",
    "waz / islamic talk": "Waz",
    "islamic lectures": "Waz",
    "islamic talk": "Waz",
    "islamic lecture (bangla dub/run)": "Waz",
    "islamic stories": "Waz",
    "speech islamik": "Waz",
    "speech": "Waz",

    # ── Talk-show ─────────────────────────────────────────────────
    "talk show": "Talk-show",
    "talkshow": "Talk-show",
    "talk-show": "Talk-show",
    "political talk show": "Talk-show",
    "reality talk show": "Talk-show",
    "entertainment talk show": "Talk-show",
    "talk & social commentary": "Talk-show",
    "live talk": "Talk-show",
    "sylheti talk": "Talk-show",
    "tv show": "Talk-show",
    "web series": "Talk-show",
    "uddokta kotha": "Talk-show",
    "orientation live": "Talk-show",
    "csrm": "Talk-show",

    # ── Podcast ───────────────────────────────────────────────────
    "personal podcast": "Podcast",
    "conversational podcast": "Podcast",
    "story podcast": "Podcast",
    "entertainment podcast": "Podcast",
    "technology podcast": "Podcast",
    "business podcast": "Podcast",
    "mental health podcast": "Podcast",
    "music podcast": "Podcast",
    "story & talk podcast": "Podcast",
    "tech & personal talk": "Podcast",

    # ── Audio-book ────────────────────────────────────────────────
    "audiobook": "Audio-book",
    "audiobook / storytelling": "Audio-book",
    "audiobook / story": "Audio-book",
    "audio-book": "Audio-book",
    "bangla puthi": "Audio-book",
    "literature discussion": "Audio-book",

    # ── Natok / Drama ─────────────────────────────────────────────
    "natok": "Natok",
    "bangla natok": "Natok",
    "short natok": "Natok",
    "comedy natok": "Natok",
    "web natok": "Natok",
    "natok / acting content": "Natok",
    "natok / telefilm": "Natok",
    "natok / short film": "Natok",
    "natok archive": "Natok",
    "drama": "Natok",
    "audio drama": "Natok",
    "tv drama": "Natok",
    "tv & stage drama": "Natok",
    "movies / drama": "Natok",
    "sylheti drama": "Natok",
    "chittagonian drama": "Natok",
    "stage drama": "Natok",

    # ── Movie ─────────────────────────────────────────────────────
    "movie": "Movie",
    "bangla cinema": "Movie",
    "bangla cinema explanation": "Movie",
    "kolkata bangla movie": "Movie",
    "movies / web series": "Movie",
    "web film": "Movie",

    # ── News ──────────────────────────────────────────────────────
    "news": "News",
    "news presentation": "News",
    "international news": "News",
    "dbc news": "News",
    "investigative & political commentary": "News",

    # ── Education ─────────────────────────────────────────────────
    "education": "Education",
    "online class": "Education",
    "online learning": "Education",
    "web development": "Education",
    "programming & tech": "Education",
    "programming education": "Education",
    "programming tutorials": "Education",
    "gk jobs": "Education",
    "bcs preli": "Education",
    "e-learning platform": "Education",

    # ── Political ─────────────────────────────────────────────────
    "political commentary / analysis": "Political",
    "political analysis": "Political",
    "political analysis / opinion": "Political",
    "political talk": "Political",
    "political & social talk": "Political",
    "debate": "Political",
    "regional language debate": "Political",
    "parliament session": "Political",

    # ── Music ─────────────────────────────────────────────────────
    "music": "Music",
    "music & talk": "Music",
    "drama & music": "Music",
    "sylheti music": "Music",
    "kirton": "Music",
    "palagan": "Music",
    "poem": "Music",
    "poetry": "Music",
    "magic bauliana": "Music",

    # ── Vlog ──────────────────────────────────────────────────────
    "food vlog": "Vlog",
    "tour vlog": "Vlog",
    "vlogs & informal talk": "Vlog",
    "informal discussion": "Vlog",
    "food & casual talk": "Vlog",

    # ── Comedy / Satire ───────────────────────────────────────────
    "satire / social commentary": "Comedy / Satire",
    "satire / talk": "Comedy / Satire",
    "comedy & acting": "Comedy / Satire",
    "rangpuri comedy": "Comedy / Satire",
    "zatrapala": "Comedy / Satire",
    "ha show": "Comedy / Satire",
    "what a show": "Comedy / Satire",

    # ── Cartoon ───────────────────────────────────────────────────
    "cartoon": "Cartoon",

    # ── Commentary ────────────────────────────────────────────────
    "bangla comentary": "Commentary",
    "commentary": "Commentary",
    "bangla commentary": "Commentary",
    "local dialect commentary": "Commentary",
    "banglabid": "Commentary",
    "lux superstar": "Commentary",

    # ── Regional / Dialect ────────────────────────────────────────
    "barishal dialect": "Regional / Dialect",
    "noakhali dialect talk": "Regional / Dialect",
    "chittagonian": "Regional / Dialect",

    # ── Health ────────────────────────────────────────────────────
    "health talk": "Health",
    "health tips": "Health",

    # ── Service / Promo ───────────────────────────────────────────
    "service promotion": "Service / Promo",
}

df = pd.read_csv(INPUT_CSV)
before = df["category"].value_counts().to_dict()

df["category"] = (
    df["category"]
    .fillna("")
    .str.strip()
    .str.lower()
    .map(lambda x: NORM.get(x, x.title() if x else "Unknown"))
)

# ── Summary ──────────────────────────────────────────────────────
after = df["category"].value_counts()
print("=" * 50)
print(f"  NORMALIZED CATEGORIES  (n={len(df)})")
print("=" * 50)
print(f"{'Category':<35} {'Count':>6}  {'%':>5}")
print("-" * 50)
for cat, cnt in after.items():
    print(f"{cat:<35} {cnt:>6}  {cnt/len(df)*100:>4.1f}%")
print("-" * 50)
print(f"Distinct categories: {len(after)}")

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
df.to_excel(OUTPUT_XLSX, index=False)
print(f"\nSaved: {OUTPUT_CSV} and {OUTPUT_XLSX}")

  NORMALIZED CATEGORIES  (n=1019)
Category                             Count      %
--------------------------------------------------
Audio-book                             277  27.2%
Talk-show                              176  17.3%
Natok                                   95   9.3%
Movie                                   74   7.3%
Waz                                     66   6.5%
Podcast                                 62   6.1%
Comedy / Satire                         43   4.2%
Vlog                                    42   4.1%
Education                               39   3.8%
Music                                   37   3.6%
News                                    35   3.4%
Political                               29   2.8%
Cartoon                                 23   2.3%
Commentary                              13   1.3%
Regional / Dialect                       4   0.4%
Health                                   2   0.2%
Service / Promo                          2   0.2%
---------------